# Phase 1: Silver Fine-Tuning - Smoke Test

**Goal:** Validate Longformer + Weighted BCE on a small sample before full RunPod run.

**Testing:** Two global attention variants:
- Variant A: [CLS] only (baseline)
- Variant B: [CLS] + first topic token

**Dataset:** ~10k stratified sample, 1-2 epochs

See full plan: `docs/plans/2026-02-03-silver-finetuning-phase1.md`

## Task 1: Environment Setup & W&B Configuration

In [24]:
# Imports
import os
import torch
import psycopg2
import pandas as pd
import numpy as np
import wandb
import json
from dotenv import load_dotenv
from tqdm.auto import tqdm
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, classification_report
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from transformers import LongformerTokenizerFast, LongformerForSequenceClassification
from torch.utils.data import Dataset, DataLoader, Subset
from torch.cuda.amp import autocast, GradScaler
import gc

load_dotenv()

# Verify GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: NVIDIA GeForce RTX 4070 Ti SUPER
VRAM: 17.2 GB


In [25]:
# ============================================================
# CONFIGURATION - Change these for different experiment variants
# ============================================================

CONFIG = {
    "model": "allenai/longformer-base-4096",
    "dataset_size": 10000,
    "max_length": 2048,
    "batch_size": 2,
    "grad_accum_steps": 8,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "epochs": 2,
    "loss": "weighted_bce",
    # CHANGE THIS FOR VARIANT B: "cls_plus_topic"
    "global_attention": "cls_plus_topic",
}

# Run name based on global attention variant
RUN_NAME = f"smoke-test-{CONFIG['global_attention'].replace('_', '-')}"
SAVE_DIR = f"saved_models/silver_smoke_test_{CONFIG['global_attention']}"

print(f"Run name: {RUN_NAME}")
print(f"Save directory: {SAVE_DIR}")

Run name: smoke-test-cls-plus-topic
Save directory: saved_models/silver_smoke_test_cls_plus_topic


In [26]:
# ============================================================
# Initialize W&B Run
# ============================================================
# Using the pattern from W&B quickstart docs

run = wandb.init(
    entity="ryrousseau-london-school-of-economics-and-political-science",
    project="frame-delta",
    name=RUN_NAME,
    config=CONFIG  # Pass our CONFIG dict - this gets tracked in W&B
)

# Note: We'll use CONFIG['key'] throughout the notebook for clarity
# wandb.config can also be accessed but CONFIG dict is simpler
print(f"W&B run initialized: {run.name}")
print(f"View at: {run.url}")

W&B run initialized: smoke-test-cls-plus-topic
View at: https://wandb.ai/ryrousseau-london-school-of-economics-and-political-science/frame-delta/runs/k50vofhe


## Task 2: Load Data from Database

In [27]:
# Connect and load stratified sample
conn = psycopg2.connect(
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT")
)
cur = conn.cursor()

# Set seed for reproducibility
cur.execute("SELECT setseed(0.42)")

# Load sample with topic for injection
cur.execute(f"""
    SELECT a.text_generic_frame, a.gpt_topic, a.title, b.maintext
    FROM mm_framing_full a
    JOIN newsarticles b ON a.url = b.url
    WHERE b.maintext IS NOT NULL
    AND LENGTH(b.maintext) > 100
    ORDER BY RANDOM()
    LIMIT {CONFIG['dataset_size']}
""")

result = cur.fetchall()
cur.close()
conn.close()

df = pd.DataFrame(result, columns=["text_generic_frame", "gpt_topic", "title", "article_text"])
print(f"Loaded {len(df):,} articles")
df.head()

Loaded 10,000 articles


,text_generic_frame,gpt_topic,title,article_text
0,"[Fairness and equality, Policy prescription an...",Sports,"Yankees, Blue Jays coaches get into petty war ...",The Toronto Blue Jays voiced their displeasure...
1,[Other],Sports,Rangers lose to Houston Astros in Game 5 of ALCS,Astros fans celebrate the Astros' third win af...
2,"[Cultural identity, Fairness and equality, Cap...",Legal,Russell Island community shaken by fire that k...,Everyone who saw the fire talks about how quic...
3,"[Quality of life, Other]",Sports,Both CU and CSU to make NCAA Tournament appear...,Both CU men's and women's basketball teams are...
4,"[Crime and punishment, Health and safety, Poli...",Health,NYC's chief medical examiner explains the all-...,Special Narcotics Prosecutor's Office statisti...


## Task 3: Preprocess Text with Topic Injection

In [28]:
# Data preprocessing (matching v2 notebook approach)

# Create word count column
df['num_words'] = df['article_text'].str.split().str.len()

print(f"Original rows: {len(df)}")

# Filter based on length (>100 words)
df = df[(df['num_words'] > 100)]
df = df.dropna()
df = df.reset_index(drop=True)

print(f"After length filter and dropna: {len(df)}")

# Keep rows only where the list is NOT exactly ['Other']
# df = df[df['text_generic_frame'].apply(lambda x: x != ['Other'])]
# df = df.reset_index(drop=True)
# print(f"After removing Other-only rows: {len(df)}")

# Engineer the text column (topic injection)
# Adding the title
df['article_text'] = df['title'] + "\n" + df['article_text']

# Adding the topic at the very start
df['article_text'] = "TOPIC:" + df['gpt_topic'] + "\n" + df['article_text']

# Preview
print(f"\nSample text (first 500 chars):")
print("-" * 50)
print(df.iloc[0]['article_text'][:500])

Original rows: 10000
After length filter and dropna: 7576

Sample text (first 500 chars):
--------------------------------------------------
TOPIC:Sports
Yankees, Blue Jays coaches get into petty war of words following sign-stealing drama
The Toronto Blue Jays voiced their displeasure of where the New York Yankees' base coaches were standing following their reported assertion that Aaron Judge was looking at or near his dugout to get information on possible tipped pitches.
Well, it did not take long for the coach drama to play itself out again at Rogers Centre on Tuesday night.
Toronto pitching coach Pete Walker yelled at Yankees thir


## Task 4: Encode Labels & Calculate Class Weights

In [29]:
official_labels = [
    "Economic", "Capacity and resources", "Morality", "Fairness and equality",
    "Legality, constitutionality and jurisprudence", "Policy prescription and evaluation",
    "Crime and punishment", "Security and defense", "Health and safety",
    "Quality of life", "Cultural identity", "Public opinion", "Political",
    "External regulation and reputation", "Other"
]

mlb = MultiLabelBinarizer(classes=official_labels)
labels_matrix = mlb.fit_transform(df['text_generic_frame'])
print(f"Labels shape: {labels_matrix.shape}")

# Show class distribution
print("\nClass distribution:")
for i, label in enumerate(official_labels):
    count = labels_matrix[:, i].sum()
    print(f"  {label:<45} {count:>5} ({100*count/len(labels_matrix):.1f}%)")

Labels shape: (7576, 15)

Class distribution:
  Economic                                       3088 (40.8%)
  Capacity and resources                         1146 (15.1%)
  Morality                                        677 (8.9%)
  Fairness and equality                          2217 (29.3%)
  Legality, constitutionality and jurisprudence  3014 (39.8%)
  Policy prescription and evaluation             3444 (45.5%)
  Crime and punishment                           3046 (40.2%)
  Security and defense                           2040 (26.9%)
  Health and safety                              2167 (28.6%)
  Quality of life                                3651 (48.2%)
  Cultural identity                              2131 (28.1%)
  Public opinion                                 2811 (37.1%)
  Political                                      3000 (39.6%)
  External regulation and reputation             1655 (21.8%)
  Other                                          1094 (14.4%)


In [30]:
# Calculate normalized inverse frequency for pos_weight
num_positives = labels_matrix.sum(axis=0)
inv_freq = 1.0 / (num_positives + 1e-5)
alpha = inv_freq / inv_freq.sum()  # Normalized inverse frequency
pos_weight = torch.tensor(alpha * len(official_labels), dtype=torch.float).to(device)

print("\nClass weights (pos_weight) - normalized inverse frequency:")
for name, weight in zip(official_labels, pos_weight.cpu().numpy()):
    print(f"  {name:<45} {weight:.4f}")


Class weights (pos_weight) - normalized inverse frequency:
  Economic                                      0.6089
  Capacity and resources                        1.6408
  Morality                                      2.7775
  Fairness and equality                         0.8481
  Legality, constitutionality and jurisprudence 0.6239
  Policy prescription and evaluation            0.5460
  Crime and punishment                          0.6173
  Security and defense                          0.9217
  Health and safety                             0.8677
  Quality of life                               0.5150
  Cultural identity                             0.8824
  Public opinion                                0.6689
  Political                                     0.6268
  External regulation and reputation            1.1362
  Other                                         1.7188


## Task 5: Tokenization with Longformer

In [31]:
# Initialize tokenizer (using Fast tokenizer like v2)
tokenizer = LongformerTokenizerFast.from_pretrained(CONFIG['model'])
print(f"Tokenizer vocab size: {tokenizer.vocab_size:,}")

# Measure token lengths to verify max_length setting (like v2)
print("\nMeasuring token lengths...")
texts = df['article_text'].tolist()
encodings_check = tokenizer(texts, add_special_tokens=True, return_attention_mask=False)
token_lens = np.array([len(x) for x in encodings_check['input_ids']])

p95 = np.percentile(token_lens, 95)
p99 = np.percentile(token_lens, 99)

print(f"Mean Length: {np.mean(token_lens):.1f}")
print(f"95th Percentile: {p95:.1f} tokens")
print(f"99th Percentile: {p99:.1f} tokens")
print(f"Max Length found: {np.max(token_lens)} tokens")
print(f"\nUsing max_length={CONFIG['max_length']} (covers {100*(token_lens <= CONFIG['max_length']).mean():.1f}% of samples)")

# Clean up the check encodings
del encodings_check, token_lens
gc.collect()

Tokenizer vocab size: 50,265

Measuring token lengths...
Mean Length: 741.9
95th Percentile: 1448.0 tokens
99th Percentile: 2101.2 tokens
Max Length found: 17700 tokens

Using max_length=2048 (covers 99.0% of samples)


129

In [32]:
# Inspect tokenization of topic prefix to understand token positions
sample_text = "TOPIC:Capacity and resources\nSample title\nSample article text"
sample_tokens = tokenizer.encode(sample_text, add_special_tokens=True)
print("Sample tokenization (to verify global attention positions):")
for i, token_id in enumerate(sample_tokens[:10]):
    print(f"  Position {i}: {token_id} -> '{tokenizer.decode([token_id])}'")
print(f"\nPosition 4 token: '{tokenizer.decode([sample_tokens[4]])}'")
print("(This is where we apply global attention in cls_plus_topic mode)")

Sample tokenization (to verify global attention positions):
  Position 0: 0 -> '<s>'
  Position 1: 28332 -> 'TOP'
  Position 2: 2371 -> 'IC'
  Position 3: 35 -> ':'
  Position 4: 15791 -> 'Cap'
  Position 5: 18583 -> 'acity'
  Position 6: 8 -> ' and'
  Position 7: 1915 -> ' resources'
  Position 8: 50118 -> '
'
  Position 9: 47241 -> 'Sample'

Position 4 token: 'Cap'
(This is where we apply global attention in cls_plus_topic mode)


## Task 6: Dataset Class with Global Attention Mask

In [33]:
# Dataset class with on-the-fly tokenization (matching v2 approach)
class NewsArticleDataset(Dataset):
    def __init__(self, df, tokenizer, labels_matrix, max_len=2048, global_attention_mode="cls_only"):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.labels = labels_matrix
        self.global_attention_mode = global_attention_mode
        print(f"Dataset created: {len(df)} samples, global_attention_mode='{global_attention_mode}'")
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row['article_text'])
        
        # Tokenize on-the-fly (no padding here - collator handles it)
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            truncation=True,
            padding=False,  # Dynamic padding in collator
            add_special_tokens=True
        )
        
        input_ids = encoding['input_ids']
        attention_mask = encoding['attention_mask']
        
        # Create global attention mask
        # 0 = Local Attention, 1 = Global Attention
        global_attention_mask = [0] * len(input_ids)
        global_attention_mask[0] = 1  # CLS token always gets global attention
        
        if self.global_attention_mode == "cls_plus_topic":
            # Position 4 is typically the first token of the topic word
            # (after <s>, TOP, IC, :)
            if len(input_ids) > 4:
                global_attention_mask[4] = 1
        
        # Get labels
        labels_vec = self.labels[idx]

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'global_attention_mask': global_attention_mask,
            'labels': torch.tensor(labels_vec, dtype=torch.float)
        }

## Task 7: Train/Val/Test Split (Stratified)

In [34]:
N = len(labels_matrix)
X_indices = np.zeros(N)

# Split 1: Train (80%) / Temp (20%)
msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, temp_idx = next(iter(msss1.split(X_indices, labels_matrix)))

# Split 2: Val (10%) / Test (10%)
temp_labels = labels_matrix[temp_idx]
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
relative_val_idx, relative_test_idx = next(iter(msss2.split(np.zeros(len(temp_idx)), temp_labels)))

val_idx = temp_idx[relative_val_idx]
test_idx = temp_idx[relative_test_idx]

print(f"Train: {len(train_idx):,} ({100*len(train_idx)/N:.0f}%)")
print(f"Val:   {len(val_idx):,} ({100*len(val_idx)/N:.0f}%)")
print(f"Test:  {len(test_idx):,} ({100*len(test_idx)/N:.0f}%)")

Train: 6,045 (80%)
Val:   759 (10%)
Test:  772 (10%)


## Task 8: Create DataLoaders

In [35]:
# Dynamic padding collate function (matching v2 approach)
def longformer_collate_fn(batch):
    """
    Custom collator to handle dynamic padding and 512-window alignment.
    """
    # Determine the maximum length in this specific batch
    max_len = max(len(item['input_ids']) for item in batch)
    
    # Round up to nearest multiple of 512 (Longformer Window Size)
    window_size = 512
    padded_len = ((max_len + window_size - 1) // window_size) * window_size
    
    # Prepare batch lists
    input_ids_batch = []
    attention_mask_batch = []
    global_attention_mask_batch = []
    labels_batch = []
    
    pad_token_id = tokenizer.pad_token_id
    
    for item in batch:
        curr_len = len(item['input_ids'])
        pad_len = padded_len - curr_len
        
        # Pad Input IDs
        ids = item['input_ids'] + [pad_token_id] * pad_len
        
        # Pad Attention Mask (0 for padded tokens)
        mask = item['attention_mask'] + [0] * pad_len
        
        # Pad Global Attention Mask (0 for padded tokens)
        global_mask = item['global_attention_mask'] + [0] * pad_len
        
        input_ids_batch.append(ids)
        attention_mask_batch.append(mask)
        global_attention_mask_batch.append(global_mask)
        labels_batch.append(item['labels'])

    return {
        'input_ids': torch.tensor(input_ids_batch, dtype=torch.long),
        'attention_mask': torch.tensor(attention_mask_batch, dtype=torch.long),
        'global_attention_mask': torch.tensor(global_attention_mask_batch, dtype=torch.long),
        'labels': torch.stack(labels_batch)
    }

# Create full dataset (tokenization happens on-the-fly in __getitem__)
full_dataset = NewsArticleDataset(
    df, 
    tokenizer, 
    labels_matrix, 
    max_len=CONFIG['max_length'],
    global_attention_mode=CONFIG['global_attention']
)

# Create subsets using the split indices
train_dataset = Subset(full_dataset, train_idx)
val_dataset = Subset(full_dataset, val_idx)
test_dataset = Subset(full_dataset, test_idx)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=CONFIG['batch_size'], 
    shuffle=True,
    collate_fn=longformer_collate_fn,
    num_workers=0,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=CONFIG['batch_size'], 
    shuffle=False,
    collate_fn=longformer_collate_fn,
    num_workers=0,
    pin_memory=True
)
test_loader = DataLoader(
    test_dataset, 
    batch_size=CONFIG['batch_size'], 
    shuffle=False,
    collate_fn=longformer_collate_fn,
    num_workers=0,
    pin_memory=True
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Dataset created: 7576 samples, global_attention_mode='cls_plus_topic'
Train batches: 3023
Val batches: 380
Test batches: 386


In [36]:
# Verify batch structure
sample_batch = next(iter(train_loader))
print(f"Sample batch shapes:")
print(f"  input_ids: {sample_batch['input_ids'].shape}")
print(f"  attention_mask: {sample_batch['attention_mask'].shape}")
print(f"  global_attention_mask: {sample_batch['global_attention_mask'].shape}")
print(f"  labels: {sample_batch['labels'].shape}")

print(f"\nGlobal attention positions (first sample):")
global_positions = (sample_batch['global_attention_mask'][0] == 1).nonzero().squeeze().tolist()
if isinstance(global_positions, int):
    global_positions = [global_positions]
print(f"  Positions with global attention: {global_positions}")

# Verify padding is aligned to 512
seq_len = sample_batch['input_ids'].shape[1]
print(f"\nSequence length in batch: {seq_len} (aligned to 512: {seq_len % 512 == 0})")

Sample batch shapes:
  input_ids: torch.Size([2, 1536])
  attention_mask: torch.Size([2, 1536])
  global_attention_mask: torch.Size([2, 1536])
  labels: torch.Size([2, 15])

Global attention positions (first sample):
  Positions with global attention: [0, 4]

Sequence length in batch: 1536 (aligned to 512: True)


## Task 9: Model Setup

In [37]:
# Clear GPU memory
import gc
gc.collect()
torch.cuda.empty_cache()

model = LongformerForSequenceClassification.from_pretrained(
    CONFIG['model'],
    num_labels=len(official_labels),
    problem_type="multi_label_classification"
)
model.to(device)

# Enable gradient checkpointing for VRAM efficiency
model.gradient_checkpointing_enable()

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=CONFIG['learning_rate'], 
    weight_decay=CONFIG['weight_decay']
)

# Loss function with class weights
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Some weights of LongformerForSequenceClassification were not initialized from the model checkpoint at allenai/longformer-base-4096 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model parameters: 148,670,991
Trainable parameters: 148,670,991


## Task 10: Training Loop

In [38]:
scaler = GradScaler()
best_val_f1 = 0.0
os.makedirs(SAVE_DIR, exist_ok=True)

# Save config
with open(f"{SAVE_DIR}/config.json", 'w') as f:
    json.dump(CONFIG, f, indent=4)
print(f"Config saved to {SAVE_DIR}/config.json")

for epoch in range(CONFIG['epochs']):
    # ==================== TRAINING ====================
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch} [Train]")
    for step, batch in pbar:
        # Move to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        global_attention_mask = batch['global_attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass with mixed precision
        with autocast():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                global_attention_mask=global_attention_mask
            )
            loss = criterion(outputs.logits, labels)
            loss = loss / CONFIG['grad_accum_steps']

        # Backward pass
        scaler.scale(loss).backward()
        train_loss += loss.item() * CONFIG['grad_accum_steps']

        # Gradient accumulation step
        if (step + 1) % CONFIG['grad_accum_steps'] == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        pbar.set_postfix({'loss': f'{loss.item() * CONFIG["grad_accum_steps"]:.4f}'})

    avg_train_loss = train_loss / len(train_loader)

    # ==================== VALIDATION ====================
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch} [Val]"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            global_attention_mask = batch['global_attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                global_attention_mask=global_attention_mask
            )

            loss = criterion(outputs.logits, labels)
            val_loss += loss.item()

            preds = (torch.sigmoid(outputs.logits) > 0.5).float()
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    avg_val_loss = val_loss / len(val_loader)
    val_f1_micro = f1_score(all_labels, all_preds, average='micro')
    val_f1_macro = f1_score(all_labels, all_preds, average='macro')

    # Log to W&B using run.log()
    run.log({
        "epoch": epoch,
        "train_loss": avg_train_loss,
        "val_loss": avg_val_loss,
        "val_f1_micro": val_f1_micro,
        "val_f1_macro": val_f1_macro,
    })

    print(f"\nEpoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"  Val F1 Micro: {val_f1_micro:.4f} | Val F1 Macro: {val_f1_macro:.4f}")

    # Checkpoint on best
    if val_f1_micro > best_val_f1:
        best_val_f1 = val_f1_micro
        model.save_pretrained(f"{SAVE_DIR}/best_model")
        tokenizer.save_pretrained(f"{SAVE_DIR}/best_model")
        # Track best in W&B summary
        run.summary["best_val_f1_micro"] = best_val_f1
        print(f"  -> New best model saved! (F1 Micro: {best_val_f1:.4f})")

print(f"\nTraining complete. Best Val F1 Micro: {best_val_f1:.4f}")

C:\Users\rhrou\AppData\Local\Temp\ipykernel_48292\3701168160.py:1: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Config saved to saved_models/silver_smoke_test_cls_plus_topic/config.json


Epoch 0 [Train]:   0%|          | 0/3023 [00:00<?, ?it/s]C:\Users\rhrou\AppData\Local\Temp\ipykernel_48292\3701168160.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 0 [Val]: 100%|██████████| 380/380 [00:32<00:00, 11.80it/s]



Epoch 0 | Train Loss: 0.4430 | Val Loss: 0.3741
  Val F1 Micro: 0.6020 | Val F1 Macro: 0.5762
  -> New best model saved! (F1 Micro: 0.6020)


Epoch 1 [Train]:   0%|          | 0/3023 [00:00<?, ?it/s]C:\Users\rhrou\AppData\Local\Temp\ipykernel_48292\3701168160.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 1 [Val]: 100%|██████████| 380/380 [00:32<00:00, 11.85it/s]



Epoch 1 | Train Loss: 0.3685 | Val Loss: 0.3553
  Val F1 Micro: 0.6511 | Val F1 Macro: 0.6279
  -> New best model saved! (F1 Micro: 0.6511)

Training complete. Best Val F1 Micro: 0.6511


## Task 11: Post-Training Threshold Optimization

In [39]:
# Load best model
print(f"Loading best model from {SAVE_DIR}/best_model...")
model = LongformerForSequenceClassification.from_pretrained(f"{SAVE_DIR}/best_model")
model.to(device)
model.eval()

# Get raw probabilities on validation set
val_probs = []
val_labels_list = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Getting probabilities"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        global_attention_mask = batch['global_attention_mask'].to(device)
        labels = batch['labels']

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            global_attention_mask=global_attention_mask
        )
        probs = torch.sigmoid(outputs.logits)

        val_probs.append(probs.cpu().numpy())
        val_labels_list.append(labels.numpy())

val_probs = np.vstack(val_probs)
val_labels_arr = np.vstack(val_labels_list)

print(f"Validation probs shape: {val_probs.shape}")
print(f"Validation labels shape: {val_labels_arr.shape}")

Loading best model from saved_models/silver_smoke_test_cls_plus_topic/best_model...


Getting probabilities: 100%|██████████| 380/380 [00:31<00:00, 11.90it/s]

Validation probs shape: (759, 15)
Validation labels shape: (759, 15)


In [40]:
# Grid search for optimal thresholds per class
best_thresholds = {}
print("\nOptimizing thresholds per class...")
print("=" * 70)

for i, label_name in enumerate(official_labels):
    best_score = 0
    best_thresh = 0.5

    y_true = val_labels_arr[:, i]
    y_score = val_probs[:, i]

    for thresh in np.arange(0.10, 0.95, 0.05):
        y_pred = (y_score > thresh).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)

        if score > best_score:
            best_score = score
            best_thresh = thresh

    best_thresholds[label_name] = float(best_thresh)
    print(f"{label_name:<50} Threshold: {best_thresh:.2f} (F1: {best_score:.3f})")

# Save thresholds
threshold_path = f"{SAVE_DIR}/class_thresholds_optimized.json"
with open(threshold_path, 'w') as f:
    json.dump(best_thresholds, f, indent=4)
print(f"\nThresholds saved to {threshold_path}")


Optimizing thresholds per class...
Economic                                           Threshold: 0.20 (F1: 0.760)
Capacity and resources                             Threshold: 0.20 (F1: 0.529)
Morality                                           Threshold: 0.75 (F1: 0.590)
Fairness and equality                              Threshold: 0.30 (F1: 0.711)
Legality, constitutionality and jurisprudence      Threshold: 0.45 (F1: 0.803)
Policy prescription and evaluation                 Threshold: 0.20 (F1: 0.733)
Crime and punishment                               Threshold: 0.30 (F1: 0.788)
Security and defense                               Threshold: 0.45 (F1: 0.746)
Health and safety                                  Threshold: 0.50 (F1: 0.777)
Quality of life                                    Threshold: 0.20 (F1: 0.774)
Cultural identity                                  Threshold: 0.35 (F1: 0.690)
Public opinion                                     Threshold: 0.20 (F1: 0.643)
Political       

## Task 12: Final Evaluation with Optimized Thresholds

In [41]:
# Evaluate on TEST set with optimized thresholds
test_probs = []
test_labels_list = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Test evaluation"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        global_attention_mask = batch['global_attention_mask'].to(device)
        labels = batch['labels']

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            global_attention_mask=global_attention_mask
        )
        probs = torch.sigmoid(outputs.logits)

        test_probs.append(probs.cpu().numpy())
        test_labels_list.append(labels.numpy())

test_probs = np.vstack(test_probs)
test_labels_arr = np.vstack(test_labels_list)

# Apply optimized thresholds
test_preds = np.zeros_like(test_probs)
for i, label_name in enumerate(official_labels):
    thresh = best_thresholds[label_name]
    test_preds[:, i] = (test_probs[:, i] > thresh).astype(int)

# Final metrics
test_f1_micro = f1_score(test_labels_arr, test_preds, average='micro')
test_f1_macro = f1_score(test_labels_arr, test_preds, average='macro')

print(f"\n{'='*70}")
print(f"FINAL TEST RESULTS (Optimized Thresholds)")
print(f"{'='*70}")
print(f"Global Attention Mode: {CONFIG['global_attention']}")
print(f"Micro F1: {test_f1_micro:.4f}")
print(f"Macro F1: {test_f1_macro:.4f}")
print(f"\nPer-class report:")
print(classification_report(test_labels_arr, test_preds, target_names=official_labels))

Test evaluation: 100%|██████████| 386/386 [00:32<00:00, 11.83it/s]


FINAL TEST RESULTS (Optimized Thresholds)
Global Attention Mode: cls_plus_topic
Micro F1: 0.7102
Macro F1: 0.6885

Per-class report:
                                               precision    recall  f1-score   support

                                     Economic       0.71      0.80      0.75       309
                       Capacity and resources       0.44      0.76      0.56       115
                                     Morality       0.59      0.65      0.62        68
                        Fairness and equality       0.57      0.73      0.64       222
Legality, constitutionality and jurisprudence       0.81      0.73      0.77       302
           Policy prescription and evaluation       0.65      0.86      0.74       345
                         Crime and punishment       0.77      0.80      0.78       310
                         Security and defense       0.82      0.74      0.78       204
                            Health and safety       0.80      0.67      0.73      

In [42]:
# Log final metrics to W&B
run.log({
    "test_f1_micro_optimized": test_f1_micro,
    "test_f1_macro_optimized": test_f1_macro,
})

# Also store in summary for easy comparison across runs
run.summary["test_f1_micro_optimized"] = test_f1_micro
run.summary["test_f1_macro_optimized"] = test_f1_macro

# Save final results summary locally
results_summary = {
    "global_attention_mode": CONFIG['global_attention'],
    "dataset_size": CONFIG['dataset_size'],
    "epochs": CONFIG['epochs'],
    "best_val_f1_micro": float(best_val_f1),
    "test_f1_micro_optimized": float(test_f1_micro),
    "test_f1_macro_optimized": float(test_f1_macro),
    "thresholds": best_thresholds
}

with open(f"{SAVE_DIR}/results_summary.json", 'w') as f:
    json.dump(results_summary, f, indent=4)

print(f"\nResults saved to {SAVE_DIR}/results_summary.json")

# Finish the W&B run
run.finish()
print("\nW&B run finished. View results at wandb.ai")


Results saved to saved_models/silver_smoke_test_cls_plus_topic/results_summary.json


epoch,▁█
test_f1_macro_optimized,▁
test_f1_micro_optimized,▁
train_loss,█▁
val_f1_macro,▁█
val_f1_micro,▁█
val_loss,█▁
best_val_f1_micro,0.65111
epoch,1
test_f1_macro_optimized,0.68849
test_f1_micro_optimized,0.71016



W&B run finished. View results at wandb.ai


## Comparison Notes

After running both variants, compare:

| Variant | Global Attention | Test F1 Micro | Test F1 Macro |
|---------|------------------|---------------|---------------|
| A | cls_only | 0.701 | 0.668 |
| B | cls_plus_topic | 0.710 | 0.689 |

**Decision criteria:**
- If B improves both metrics by >1%: Use B for full run
- If difference is <1%: Use A (simpler, less compute)
- If B is worse: Definitely use A